In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from datetime import datetime, timedelta
import string

In [17]:
unbalanced_data = pd.read_csv('E:/tvb_25/datasets/Twitter Analysis.csv')
unbalanced_data.head()

,Unnamed: 0,majority_target,statement,BinaryNumTarget,tweet,followers_count,friends_count,favourites_count,statuses_count,listed_count,...,determiners,conjunctions,dots,exclamation,questions,ampersand,capitals,digits,long_word_freq,short_word_freq
0,0,True,End of eviction moratorium means millions of A...,1.0,@POTUS Biden Blunders - 6 Month Update\n\nInfl...,4262.0,3619.0,34945.0,16423.0,44.0,...,0,0,5,0,1,0,33,3,5,19
1,1,True,End of eviction moratorium means millions of A...,1.0,@S0SickRick @Stairmaster_ @6d6f636869 Not as m...,1393.0,1621.0,31436.0,37184.0,64.0,...,0,2,1,0,0,0,14,0,2,34
2,2,True,End of eviction moratorium means millions of A...,1.0,THE SUPREME COURT is siding with super rich pr...,9.0,84.0,219.0,1184.0,0.0,...,0,1,0,0,0,0,3,0,4,10
3,3,True,End of eviction moratorium means millions of A...,1.0,@POTUS Biden Blunders\n\nBroken campaign promi...,4262.0,3619.0,34945.0,16423.0,44.0,...,0,1,3,0,0,1,6,8,1,30
4,4,True,End of eviction moratorium means millions of A...,1.0,@OhComfy I agree. The confluence of events rig...,70.0,166.0,15282.0,2194.0,0.0,...,0,1,3,0,1,0,11,3,2,19


In [3]:
unbalanced_data.describe()

,Unnamed: 0,BinaryNumTarget,followers_count,friends_count,favourites_count,statuses_count,listed_count,following,BotScore,BotScoreBinary,...,determiners,conjunctions,dots,exclamation,questions,ampersand,capitals,digits,long_word_freq,short_word_freq
count,134198.00000,134198.000000,1.341980e+05,134198.000000,1.341980e+05,1.341980e+05,134198.000000,134198.0,134198.000000,134198.000000,...,134198.000000,134198.000000,134198.000000,134198.000000,134198.000000,134198.000000,134198.000000,134198.000000,134198.000000,134198.000000
mean,67098.50000,0.513644,1.129308e+04,1893.454455,3.298123e+04,3.419576e+04,73.300198,0.0,0.059106,0.032355,...,0.135583,1.003495,2.366116,0.259408,0.307151,0.121537,12.831905,3.559494,2.249557,21.438658
std,38739.77005,0.499816,4.374971e+05,6997.695671,6.878021e+04,7.510120e+04,1083.274277,0.0,0.167819,0.176942,...,0.379235,1.086844,2.140459,0.903957,0.774367,0.453865,15.557524,6.674458,2.912136,9.625147
min,0.00000,0.000000,0.000000e+00,0.000000,0.000000e+00,1.000000e+00,0.000000,0.0,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,33549.25000,0.000000,7.000000e+01,168.000000,1.356000e+03,3.046000e+03,0.000000,0.0,0.030000,0.000000,...,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,6.000000,0.000000,1.000000,14.000000
50%,67098.50000,1.000000,3.540000e+02,567.000000,8.377000e+03,1.101900e+04,2.000000,0.0,0.030000,0.000000,...,0.000000,1.000000,2.000000,0.000000,0.000000,0.000000,10.000000,2.000000,2.000000,21.000000
75%,100647.75000,1.000000,1.573000e+03,1726.000000,3.352650e+04,3.357375e+04,11.000000,0.0,0.030000,0.000000,...,0.000000,2.000000,3.000000,0.000000,0.000000,0.000000,15.000000,4.000000,3.000000,28.000000
max,134197.00000,1.000000,1.306019e+08,586901.000000,1.765080e+06,2.958918e+06,222193.000000,0.0,1.000000,1.000000,...,5.000000,13.000000,50.000000,66.000000,43.000000,13.000000,250.000000,138.000000,47.000000,164.000000


In [4]:
unbalanced_data.columns

Index(['Unnamed: 0', 'majority_target', 'statement', 'BinaryNumTarget',
       'tweet', 'followers_count', 'friends_count', 'favourites_count',
       'statuses_count', 'listed_count', 'following', 'embeddings', 'BotScore',
       'BotScoreBinary', 'cred', 'normalize_influence', 'mentions', 'quotes',
       'replies', 'retweets', 'favourites', 'hashtags', 'URLs', 'unique_count',
       'total_count', 'ORG_percentage', 'NORP_percentage', 'GPE_percentage',
       'PERSON_percentage', 'MONEY_percentage', 'DATE_percentage',
       'CARDINAL_percentage', 'PERCENT_percentage', 'ORDINAL_percentage',
       'FAC_percentage', 'LAW_percentage', 'PRODUCT_percentage',
       'EVENT_percentage', 'TIME_percentage', 'LOC_percentage',
       'WORK_OF_ART_percentage', 'QUANTITY_percentage', 'LANGUAGE_percentage',
       'Word count', 'Max word length', 'Min word length',
       'Average word length', 'present_verbs', 'past_verbs', 'adjectives',
       'adverbs', 'adpositions', 'pronouns', 'TOs', 'deter

In [5]:
# Changing column names so they are all equal (adding _ for spaces)
unbalanced_data.columns = unbalanced_data.columns.str.replace(' ', '_')

In [6]:
unbalanced_data.columns

Index(['Unnamed:_0', 'majority_target', 'statement', 'BinaryNumTarget',
       'tweet', 'followers_count', 'friends_count', 'favourites_count',
       'statuses_count', 'listed_count', 'following', 'embeddings', 'BotScore',
       'BotScoreBinary', 'cred', 'normalize_influence', 'mentions', 'quotes',
       'replies', 'retweets', 'favourites', 'hashtags', 'URLs', 'unique_count',
       'total_count', 'ORG_percentage', 'NORP_percentage', 'GPE_percentage',
       'PERSON_percentage', 'MONEY_percentage', 'DATE_percentage',
       'CARDINAL_percentage', 'PERCENT_percentage', 'ORDINAL_percentage',
       'FAC_percentage', 'LAW_percentage', 'PRODUCT_percentage',
       'EVENT_percentage', 'TIME_percentage', 'LOC_percentage',
       'WORK_OF_ART_percentage', 'QUANTITY_percentage', 'LANGUAGE_percentage',
       'Word_count', 'Max_word_length', 'Min_word_length',
       'Average_word_length', 'present_verbs', 'past_verbs', 'adjectives',
       'adverbs', 'adpositions', 'pronouns', 'TOs', 'deter

## For this classification task, the target variable is BinaryNumTarget
- that represents a binary value 1 = human, 2 = bot

In [7]:
# Targret variable
unbalanced_data['BinaryNumTarget'].value_counts()

BinaryNumTarget
1.0    68930
0.0    65268
Name: count, dtype: int64

In [8]:
# Check for missing values to fill / replace
unbalanced_data.isnull().sum()

Unnamed:_0         0
majority_target    0
statement          0
BinaryNumTarget    0
tweet              0
                  ..
ampersand          0
capitals           0
digits             0
long_word_freq     0
short_word_freq    0
Length: 64, dtype: int64

### Adding real twitter metadata variables to the dataset (found in Diego's branch)

In [9]:
import numpy as np

# Temporal features
temporal_features = {
    "avg_time_between_tweets_seconds": np.nan,  # Numerical
    "median_time_between_tweets_seconds": np.nan,  # Numerical
    "tweet_frequency_per_day": 0.0,  # Numerical
    "most_active_hour": 0,  # Numerical (0-23)
    "weekend_tweet_ratio": 0.0,  # Numerical (0-1)
    "max_tweets_per_hour": 0,  # Numerical
    "avg_tweets_per_active_hour": 0.0  # Numerical
}

# Engagement features
engagement_features = {
    "avg_likes_per_tweet": 0.0,  # Numerical
    "avg_retweets_per_tweet": 0.0,  # Numerical
    "avg_replies_per_tweet": 0.0,  # Numerical
    "avg_views_per_tweet": 0.0,  # Numerical
    "median_likes_per_tweet": 0.0,  # Numerical
    "median_retweets_per_tweet": 0.0,  # Numerical
    "pct_tweets_with_media": 0.0,  # Numerical (percentage)
    "pct_tweets_with_video": 0.0,  # Numerical (percentage)
    "pct_tweets_with_hashtags": 0.0,  # Numerical (percentage)
    "pct_tweets_with_urls": 0.0,  # Numerical (percentage)
    "pct_retweets": 0.0,  # Numerical (percentage)
    "pct_replies": 0.0,  # Numerical (percentage)
    "pct_tweets_with_spam_keywords": 0.0  # Numerical (percentage)
}

# Tweet-level features
tweet_features = {
    "tweet_id": [""] * len(unbalanced_data),  # String/Categorical
    "username": [""] * len(unbalanced_data),  # String/Categorical
    "handle": [""] * len(unbalanced_data),  # String/Categorical
    "tweet_text": [""] * len(unbalanced_data),  # String/Text
    "tweet_url": [""] * len(unbalanced_data),  # String/URL
    "timestamp": [""] * len(unbalanced_data),  # String/Datetime
    "source": [""] * len(unbalanced_data),  # String/Categorical
    "lang": [None] * len(unbalanced_data),  # String/Categorical
    "reply_count": 0,  # Numerical - scalar broadcasts to all rows
    "retweet_count": 0,
    "like_count": 0,
    "view_count": 0,
    "quote_count": 0,
    "bookmark_count": 0,
    "hashtag_count": 0,
    "mention_count": 0,
    "url_count": 0,
    "media_count": 0,
    "text_length": 0,
    "word_count": 0,
    "emoji_count": 0,
    "exclamation_count": 0,
    "question_count": 0,
    "uppercase_ratio": 0.0,
    "punctuation_ratio": 0.0,
    "digit_ratio": 0.0,
    "is_retweet": False,  # Boolean - scalar broadcasts
    "is_reply": False,
    "is_quote": False,
    "has_media": False,
    "has_video": False,
    "has_hashtags": False,
    "has_mentions": False,
    "has_urls": False,
    "possibly_sensitive": False,
    "contains_spam_keywords": False,
    "hashtags": [[] for _ in range(len(unbalanced_data))],  # List - need unique list per row
    "mentions": [[] for _ in range(len(unbalanced_data))],
    "urls": [[] for _ in range(len(unbalanced_data))],
    "hour_of_day": 0,
    "day_of_week": 0,
    "is_weekend": False,
    "scraped_at": [""] * len(unbalanced_data)
}

# User-level/Account metadata features
user_features = {
    "display_name": [""] * len(unbalanced_data),  # String/Categorical
    "bio": [""] * len(unbalanced_data),  # String/Text
    "location": [None] * len(unbalanced_data),  # String/Categorical (can be null)
    "website": [None] * len(unbalanced_data),  # String/URL (can be null)
    "created_at": [""] * len(unbalanced_data),  # String/Datetime
    "following_count": 0,  # Numerical
    "tweet_count": 0,  # Numerical
    "account_age_days": 0,  # Numerical
    "description_length": 0,  # Numerical
    "name_length": 0,  # Numerical
    "screen_name_length": 0,  # Numerical
    "followers_to_following_ratio": 0.0,  # Numerical
    "verified": False,  # Boolean
    "has_description": False,  # Boolean
    "has_url": False,  # Boolean
    "has_location": False,  # Boolean
    "default_profile_image": False,  # Boolean
    "profile_scraped_at": [""] * len(unbalanced_data)  # String/Datetime
}

# Combine all features
all_features = {**temporal_features, **engagement_features, **tweet_features, **user_features}

# Check for duplicates and add features
existing_columns = set(unbalanced_data.columns)
features_added = 0
features_skipped = []

for feature_name, feature_value in all_features.items():
    if feature_name not in existing_columns:
        unbalanced_data[feature_name] = feature_value
        features_added += 1
    else:
        features_skipped.append(feature_name)

print(f"✅ Added {features_added} new columns")
print(f"⚠️ Skipped {len(features_skipped)} duplicate columns: {features_skipped}")
print(f"📊 New shape: {unbalanced_data.shape}")

✅ Added 79 new columns
⚠️ Skipped 2 duplicate columns: ['hashtags', 'mentions']
📊 New shape: (134198, 143)


In [10]:
unbalanced_data.shape

(134198, 143)

In [13]:
unbalanced_data.isnull().sum()

Unnamed:_0               0
majority_target          0
statement                0
BinaryNumTarget          0
tweet                    0
                        ..
has_description          0
has_url                  0
has_location             0
default_profile_image    0
profile_scraped_at       0
Length: 143, dtype: int64

## Filling synthetic data with values

In [15]:
# Find columns with different types of "empty" values
empty_value_cols = []

for col in unbalanced_data.columns:
    # Check for actual nulls
    if unbalanced_data[col].isnull().any():
        empty_value_cols.append((col, 'null/NaN'))
    # Check for empty strings
    elif unbalanced_data[col].dtype == 'object':
        if (unbalanced_data[col] == "").any():
            empty_value_cols.append((col, 'empty string'))
    # Check for all zeros (numeric columns only)
    elif np.issubdtype(unbalanced_data[col].dtype, np.number):
        if (unbalanced_data[col] == 0).all():
            empty_value_cols.append((col, 'all zeros'))
    # Check for all False (boolean columns)
    elif unbalanced_data[col].dtype == 'bool':
        if (~unbalanced_data[col]).all():  # All False
            empty_value_cols.append((col, 'all False'))

print(f"Found {len(empty_value_cols)} columns with placeholder/empty values:\n")
for col, val_type in empty_value_cols:
    print(f"  - {col}: {val_type}")

Found 79 columns with placeholder/empty values:

  - following: all zeros
  - avg_time_between_tweets_seconds: null/NaN
  - median_time_between_tweets_seconds: null/NaN
  - tweet_frequency_per_day: all zeros
  - most_active_hour: all zeros
  - weekend_tweet_ratio: all zeros
  - max_tweets_per_hour: all zeros
  - avg_tweets_per_active_hour: all zeros
  - avg_likes_per_tweet: all zeros
  - avg_retweets_per_tweet: all zeros
  - avg_replies_per_tweet: all zeros
  - avg_views_per_tweet: all zeros
  - median_likes_per_tweet: all zeros
  - median_retweets_per_tweet: all zeros
  - pct_tweets_with_media: all zeros
  - pct_tweets_with_video: all zeros
  - pct_tweets_with_hashtags: all zeros
  - pct_tweets_with_urls: all zeros
  - pct_retweets: all zeros
  - pct_replies: all zeros
  - pct_tweets_with_spam_keywords: all zeros
  - tweet_id: empty string
  - username: empty string
  - handle: empty string
  - tweet_text: empty string
  - tweet_url: empty string
  - timestamp: empty string
  - source

## Synthetic modeling for missing features including:
- 3 segments: human, content_bot, spam_bot
- Diurnal patterns (more posts at certain hours)
- Based on followers + content + time + user type
- User-specific propensity (some users post more media)
- Evening boost, weekend effects
- Bots have distinctive patterns (post more, different times, less engagement)


# This part was written by AI as we lack the knowledge to do this.

In [29]:
import numpy as np
import pandas as pd

# Set random seed for reproducibility
np.random.seed(42)
n = len(unbalanced_data)

print("Starting synthetic data generation...")

# ============================================================
# 1. TIMESTAMPS
# ============================================================
print("Filling timestamps...")

# Scraped timestamps (within last 30 days)
scraped_mask = unbalanced_data['scraped_at'].astype(str).str.len().eq(0)
if scraped_mask.any():
    days_ago = np.random.uniform(0, 30, scraped_mask.sum())
    unbalanced_data.loc[scraped_mask, 'scraped_at'] = pd.Timestamp.now() - pd.to_timedelta(days_ago, unit='D')

# Tweet timestamps (before scraped_at)
timestamp_mask = unbalanced_data['timestamp'].astype(str).str.len().eq(0)
if timestamp_mask.any():
    days_before = np.random.uniform(0, 45, n)
    hours = np.random.randint(0, 24, n)
    unbalanced_data['timestamp'] = pd.to_datetime(unbalanced_data['scraped_at']) - pd.to_timedelta(days_before, unit='D') + pd.to_timedelta(hours, unit='h')

# Derive hour of day and day of week from timestamp
unbalanced_data['hour_of_day'] = pd.to_datetime(unbalanced_data['timestamp']).dt.hour
unbalanced_data['day_of_week'] = pd.to_datetime(unbalanced_data['timestamp']).dt.dayofweek
unbalanced_data['is_weekend'] = unbalanced_data['day_of_week'].isin([5, 6])

# ============================================================
# 2. USER PROFILE
# ============================================================
print("Filling user profiles...")

# Account creation date (1-5 years before tweet timestamp)
created_mask = unbalanced_data['created_at'].astype(str).str.len().eq(0)
if created_mask.any():
    age_days = np.random.randint(365, 1825, n)
    unbalanced_data['created_at'] = pd.to_datetime(unbalanced_data['timestamp']) - pd.to_timedelta(age_days, unit='D')
    unbalanced_data['account_age_days'] = age_days

# Profile scraped at
profile_scraped_mask = unbalanced_data['profile_scraped_at'].astype(str).str.len().eq(0)
if profile_scraped_mask.any():
    unbalanced_data.loc[profile_scraped_mask, 'profile_scraped_at'] = unbalanced_data['scraped_at']

# Usernames and handles
username_mask = unbalanced_data['username'].astype(str).str.len().eq(0)
if username_mask.any():
    unbalanced_data.loc[username_mask, 'username'] = ['user' + str(i).zfill(8) for i in range(username_mask.sum())]

handle_mask = unbalanced_data['handle'].astype(str).str.len().eq(0)
if handle_mask.any():
    unbalanced_data.loc[handle_mask, 'handle'] = '@' + unbalanced_data.loc[handle_mask, 'username'].astype(str)

display_mask = unbalanced_data['display_name'].astype(str).str.len().eq(0)
if display_mask.any():
    unbalanced_data.loc[display_mask, 'display_name'] = unbalanced_data.loc[display_mask, 'username'].astype(str).str.capitalize()

# Bio
bio_mask = unbalanced_data['bio'].astype(str).str.len().eq(0)
if bio_mask.any():
    bios = ["Tech enthusiast", "News and updates", "Sharing thoughts", "Content creator", "Just tweeting"]
    unbalanced_data.loc[bio_mask, 'bio'] = np.random.choice(bios, bio_mask.sum())

unbalanced_data['description_length'] = unbalanced_data['bio'].astype(str).str.len()
unbalanced_data['name_length'] = unbalanced_data['display_name'].astype(str).str.len()
unbalanced_data['screen_name_length'] = unbalanced_data['username'].astype(str).str.len()

# Following count
if (unbalanced_data['following_count'] == 0).all():
    unbalanced_data['following_count'] = np.random.lognormal(5, 1.5, n).astype(int).clip(10, 10000)

# Followers to following ratio
unbalanced_data['followers_to_following_ratio'] = (
    unbalanced_data['followers_count'] / unbalanced_data['following_count'].replace(0, 1)
).clip(0, 100)

# Tweet count
if (unbalanced_data['tweet_count'] == 0).all():
    unbalanced_data['tweet_count'] = (unbalanced_data['account_age_days'] * np.random.uniform(0.5, 3, n)).astype(int).clip(1)

# Verified and other booleans - Convert to bool first
unbalanced_data['verified'] = unbalanced_data['verified'].astype(bool) | (np.random.random(n) < 0.05)
unbalanced_data['has_description'] = unbalanced_data['bio'].astype(str).str.len() > 0
unbalanced_data['has_url'] = unbalanced_data['has_url'].astype(bool) | (np.random.random(n) < 0.3)
unbalanced_data['has_location'] = unbalanced_data['has_location'].astype(bool) | (np.random.random(n) < 0.4)
unbalanced_data['default_profile_image'] = unbalanced_data['default_profile_image'].astype(bool) | (np.random.random(n) < 0.15)

# Location and website
location_mask = unbalanced_data['location'].isna()
if location_mask.any():
    locations = ['New York', 'London', 'Tokyo', 'Paris', 'Berlin', None]
    unbalanced_data.loc[location_mask, 'location'] = np.random.choice(locations, location_mask.sum(), p=[0.15, 0.15, 0.1, 0.1, 0.1, 0.4])

website_mask = unbalanced_data['website'].isna()
if website_mask.any():
    websites = ['https://example.com', 'https://blog.com', None]
    unbalanced_data.loc[website_mask, 'website'] = np.random.choice(websites, website_mask.sum(), p=[0.15, 0.15, 0.7])

# ============================================================
# 3. TWEET CONTENT
# ============================================================
print("Filling tweet content...")

# Tweet IDs
tid_mask = unbalanced_data['tweet_id'].astype(str).str.len().eq(0)
if tid_mask.any():
    unbalanced_data.loc[tid_mask, 'tweet_id'] = [str(np.random.randint(10**17, 10**18)) for _ in range(tid_mask.sum())]

# Tweet URLs
url_mask = unbalanced_data['tweet_url'].astype(str).str.len().eq(0)
if url_mask.any():
    unbalanced_data.loc[url_mask, 'tweet_url'] = 'https://x.com/' + unbalanced_data.loc[url_mask, 'username'].astype(str) + '/status/' + unbalanced_data.loc[url_mask, 'tweet_id'].astype(str)

# Tweet text
text_mask = unbalanced_data['tweet_text'].astype(str).str.len().eq(0)
if text_mask.any():
    unbalanced_data.loc[text_mask, 'tweet_text'] = 'This is tweet number ' + unbalanced_data.loc[text_mask, 'tweet_id'].astype(str)

# Language
lang_mask = unbalanced_data['lang'].isna()
if lang_mask.any():
    langs = ['en', 'es', 'pt', 'fr', 'de', 'ja', 'ar']
    unbalanced_data.loc[lang_mask, 'lang'] = np.random.choice(langs, lang_mask.sum(), p=[0.6, 0.15, 0.08, 0.07, 0.05, 0.03, 0.02])

# Source
source_mask = unbalanced_data['source'].astype(str).str.len().eq(0)
if source_mask.any():
    sources = ['Twitter for iPhone', 'Twitter for Android', 'Twitter Web App', 'Hootsuite', 'Buffer']
    unbalanced_data.loc[source_mask, 'source'] = np.random.choice(sources, source_mask.sum(), p=[0.40, 0.35, 0.20, 0.03, 0.02])

# Content flags - Convert to bool first
unbalanced_data['has_media'] = unbalanced_data['has_media'].astype(bool) | (np.random.random(n) < 0.4)
unbalanced_data['has_video'] = unbalanced_data['has_video'].astype(bool) | (unbalanced_data['has_media'] & (np.random.random(n) < 0.3))
unbalanced_data['has_hashtags'] = unbalanced_data['has_hashtags'].astype(bool) | (np.random.random(n) < 0.5)
unbalanced_data['has_mentions'] = unbalanced_data['has_mentions'].astype(bool) | (np.random.random(n) < 0.6)
unbalanced_data['has_urls'] = unbalanced_data['has_urls'].astype(bool) | (np.random.random(n) < 0.3)

# Tweet type (mutually exclusive) - Convert to bool first
unbalanced_data['is_retweet'] = unbalanced_data['is_retweet'].astype(bool)
unbalanced_data['is_reply'] = unbalanced_data['is_reply'].astype(bool)
unbalanced_data['is_quote'] = unbalanced_data['is_quote'].astype(bool)

original_types = (unbalanced_data['is_retweet'] == False) & (unbalanced_data['is_reply'] == False) & (unbalanced_data['is_quote'] == False)
if original_types.any():
    tweet_types = np.random.choice(['original', 'retweet', 'reply', 'quote'], original_types.sum(), p=[0.7, 0.15, 0.10, 0.05])
    unbalanced_data.loc[original_types, 'is_retweet'] = tweet_types == 'retweet'
    unbalanced_data.loc[original_types, 'is_reply'] = tweet_types == 'reply'
    unbalanced_data.loc[original_types, 'is_quote'] = tweet_types == 'quote'

# Other flags - Convert to bool first
unbalanced_data['possibly_sensitive'] = unbalanced_data['possibly_sensitive'].astype(bool) | (np.random.random(n) < 0.05)
unbalanced_data['contains_spam_keywords'] = unbalanced_data['contains_spam_keywords'].astype(bool) | (np.random.random(n) < 0.02)

# ============================================================
# 4. TEXT METRICS
# ============================================================
print("Filling text metrics...")

if (unbalanced_data['text_length'] == 0).all():
    unbalanced_data['text_length'] = np.random.randint(20, 280, n)

if (unbalanced_data['word_count'] == 0).all():
    unbalanced_data['word_count'] = (unbalanced_data['text_length'] / 5).astype(int).clip(1)

if (unbalanced_data['emoji_count'] == 0).all():
    unbalanced_data['emoji_count'] = np.random.poisson(0.3, n)
if (unbalanced_data['exclamation_count'] == 0).all():
    unbalanced_data['exclamation_count'] = np.random.poisson(0.2, n)
if (unbalanced_data['question_count'] == 0).all():
    unbalanced_data['question_count'] = np.random.poisson(0.1, n)

if (unbalanced_data['uppercase_ratio'] == 0).all():
    unbalanced_data['uppercase_ratio'] = np.random.beta(2, 20, n).clip(0, 1)
if (unbalanced_data['punctuation_ratio'] == 0).all():
    unbalanced_data['punctuation_ratio'] = np.random.beta(3, 30, n).clip(0, 1)
if (unbalanced_data['digit_ratio'] == 0).all():
    unbalanced_data['digit_ratio'] = np.random.beta(2, 50, n).clip(0, 1)

# Counts based on flags
if (unbalanced_data['hashtag_count'] == 0).all():
    unbalanced_data['hashtag_count'] = np.where(unbalanced_data['has_hashtags'], np.random.poisson(1.5, n), 0)
if (unbalanced_data['mention_count'] == 0).all():
    unbalanced_data['mention_count'] = np.where(unbalanced_data['has_mentions'], np.random.poisson(1.3, n), 0)
if (unbalanced_data['url_count'] == 0).all():
    unbalanced_data['url_count'] = np.where(unbalanced_data['has_urls'], np.random.poisson(1.0, n), 0)
if (unbalanced_data['media_count'] == 0).all():
    unbalanced_data['media_count'] = np.where(unbalanced_data['has_media'], np.random.poisson(1.0, n), 0)

# ============================================================
# 5. ENGAGEMENT METRICS
# ============================================================
print("Filling engagement metrics...")

# Base engagement on followers
base_engagement = (unbalanced_data['followers_count'] * 0.01).clip(1)

if (unbalanced_data['view_count'] == 0).all():
    unbalanced_data['view_count'] = (base_engagement * np.random.lognormal(3, 1.5, n)).astype(int).clip(0)

if (unbalanced_data['like_count'] == 0).all():
    unbalanced_data['like_count'] = (unbalanced_data['view_count'] * np.random.beta(2, 100, n) * 0.15).astype(int)

if (unbalanced_data['retweet_count'] == 0).all():
    unbalanced_data['retweet_count'] = (unbalanced_data['like_count'] * np.random.beta(1, 10, n) * 0.2).astype(int)

if (unbalanced_data['reply_count'] == 0).all():
    unbalanced_data['reply_count'] = (unbalanced_data['like_count'] * np.random.beta(1, 15, n) * 0.15).astype(int)

if (unbalanced_data['quote_count'] == 0).all():
    unbalanced_data['quote_count'] = (unbalanced_data['retweet_count'] * np.random.beta(1, 20, n) * 0.1).astype(int)

if (unbalanced_data['bookmark_count'] == 0).all():
    unbalanced_data['bookmark_count'] = (unbalanced_data['like_count'] * np.random.beta(1, 30, n) * 0.05).astype(int)

# ============================================================
# 6. AGGREGATE FEATURES (USER-LEVEL STATS)
# ============================================================
print("Calculating user-level aggregates...")

# Calculate per-user aggregates
user_stats = unbalanced_data.groupby('username').agg({
    'like_count': ['mean', 'median'],
    'retweet_count': ['mean', 'median'],
    'reply_count': 'mean',
    'view_count': 'mean',
    'has_media': 'mean',
    'has_video': 'mean',
    'has_hashtags': 'mean',
    'has_urls': 'mean',
    'is_retweet': 'mean',
    'is_reply': 'mean',
    'contains_spam_keywords': 'mean',
    'hour_of_day': lambda x: x.mode()[0] if len(x.mode()) > 0 else 12,
    'is_weekend': 'mean',
    'timestamp': ['min', 'max', 'count']
}).reset_index()

# Flatten column names
user_stats.columns = ['username', 'avg_likes_per_tweet', 'median_likes_per_tweet',
                      'avg_retweets_per_tweet', 'median_retweets_per_tweet',
                      'avg_replies_per_tweet', 'avg_views_per_tweet',
                      'pct_tweets_with_media', 'pct_tweets_with_video',
                      'pct_tweets_with_hashtags', 'pct_tweets_with_urls',
                      'pct_retweets', 'pct_replies', 'pct_tweets_with_spam_keywords',
                      'most_active_hour', 'weekend_tweet_ratio',
                      'first_tweet', 'last_tweet', 'tweet_count_calc']

# Calculate temporal features
user_stats['tweet_frequency_per_day'] = user_stats['tweet_count_calc'] / (
    (user_stats['last_tweet'] - user_stats['first_tweet']).dt.days.clip(lower=1)
)
user_stats['avg_time_between_tweets_seconds'] = (
    (user_stats['last_tweet'] - user_stats['first_tweet']).dt.total_seconds() / user_stats['tweet_count_calc'].clip(lower=1)
)
user_stats['median_time_between_tweets_seconds'] = user_stats['avg_time_between_tweets_seconds'] * 0.8
user_stats['max_tweets_per_hour'] = np.random.randint(1, 10, len(user_stats))
user_stats['avg_tweets_per_active_hour'] = user_stats['tweet_frequency_per_day'] / 24

# Drop old aggregate columns if they exist
agg_cols = ['avg_likes_per_tweet', 'median_likes_per_tweet',
            'avg_retweets_per_tweet', 'median_retweets_per_tweet',
            'avg_replies_per_tweet', 'avg_views_per_tweet',
            'pct_tweets_with_media', 'pct_tweets_with_video',
            'pct_tweets_with_hashtags', 'pct_tweets_with_urls',
            'pct_retweets', 'pct_replies', 'pct_tweets_with_spam_keywords',
            'most_active_hour', 'weekend_tweet_ratio',
            'tweet_frequency_per_day', 'avg_time_between_tweets_seconds',
            'median_time_between_tweets_seconds', 'max_tweets_per_hour',
            'avg_tweets_per_active_hour']

unbalanced_data = unbalanced_data.drop(columns=agg_cols, errors='ignore')

# Drop temporary columns from user_stats
user_stats = user_stats.drop(columns=['first_tweet', 'last_tweet', 'tweet_count_calc'])

# Merge back to main dataframe
unbalanced_data = unbalanced_data.merge(user_stats, on='username', how='left')

print(f"\n✅ Synthetic data filling complete!")
print(f"📊 Final shape: {unbalanced_data.shape}")
print(f"🔍 Checking for remaining nulls...")
print(unbalanced_data.isnull().sum().sum(), "total null values remaining")

Starting synthetic data generation...
Filling timestamps...
Filling user profiles...
Filling tweet content...
Filling text metrics...
Filling engagement metrics...
Calculating user-level aggregates...

✅ Synthetic data filling complete!
📊 Final shape: (134198, 142)
🔍 Checking for remaining nulls...
4160138 total null values remaining


In [32]:
null_cols = unbalanced_data.columns[unbalanced_data.isnull().any()].tolist()
print(null_cols)

['tweet_id', 'tweet_text', 'tweet_url', 'source', 'reply_count', 'retweet_count', 'like_count', 'view_count', 'quote_count', 'bookmark_count', 'hashtag_count', 'mention_count', 'url_count', 'media_count', 'text_length', 'word_count', 'emoji_count', 'exclamation_count', 'question_count', 'uppercase_ratio', 'punctuation_ratio', 'digit_ratio', 'following_count', 'tweet_count', 'followers_to_following_ratio', 'avg_likes_per_tweet', 'median_likes_per_tweet', 'avg_retweets_per_tweet', 'median_retweets_per_tweet', 'avg_replies_per_tweet', 'avg_views_per_tweet']


## Verifying DataTypes are correct for modelling


## Showcase Distribution of Humans and Bots
- Bell curves accounting for variables
- Find patterns that humans have that bots dont

## Variable hierarchy, importance in relation to our tv: BinaryNumTarget

## Drop weak variables

## Train Model (LightGBM / XGboost)

## Visualize results
- Is there any patterns that humans have that bots dont?
- Important variables for prediction
- Pie charts, histograms, scatterplots
- How can we interpret these results in a business context?
- **Time it takes to run model**